# 02 — Quantization Analysis: PTQ vs QAT

Comparativa cuantitativa de la degradación geométrica bajo PTQ y QAT.
Ejecuta los 5 bloques de evaluación y visualiza los resultados en tablas y gráficos.

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
import torch
import pandas as pd
import matplotlib.pyplot as plt

from geoquant.models.backbone import build_backbone
from geoquant.data.dataset import get_dataloaders
from geoquant.evaluation.suite import EvaluationSuite
from geoquant.evaluation.reporter import Reporter
from geoquant.utils.reproducibility import seed_everything

seed_everything(42)

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cpu')
_, val_loader = get_dataloaders(config)

In [ ]:
def extract_embeddings(ckpt_path):
    model = build_backbone(config)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    embs, labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            embs.append(model(x).cpu())
            labels.append(y)
    return torch.cat(embs), torch.cat(labels)

emb_fp32, labels = extract_embeddings('../outputs/checkpoints/best_fp32.pth')
emb_ptq, _ = extract_embeddings('../outputs/checkpoints/mobilenet_v3_small_ptq_int8.pth')
emb_qat, _ = extract_embeddings('../outputs/checkpoints/mobilenet_v3_small_qat_int8.pth')

In [ ]:
suite = EvaluationSuite(k_neighbors=10)
results_ptq = suite.run(emb_fp32, emb_ptq, labels)
results_qat = suite.run(emb_fp32, emb_qat, labels)

reporter = Reporter('../outputs/results')
reporter.save_all(results_ptq, stem='eval_ptq')
reporter.save_all(results_qat, stem='eval_qat')

print('Resultados guardados en outputs/results/')